In [ ]:
import pandas as pd
import os
import re
from datetime import datetime
import openpyxl  # Motor recomendado para escribir archivos Excel (.xlsx)
import glob

R_Pila_I_SIE = r"\\Servernas\AYC2\(01. ASEGURAMIENTO)\eps_data_management\SIE\Pila_SIE\Pila I"
R_Pila_IP_SIE = r"\\Servernas\AYC2\(01. ASEGURAMIENTO)\eps_data_management\SIE\Pila_SIE\Pila IP"
R_Pila3047 = r"\\Servernas\AYC2\(01. ASEGURAMIENTO)\eps_data_management\Procesos BDUA\Contributivo\Compensación\Pila consiliada ADRES\Pila_Unificado_Con_Aportante_2018_2026.TXT"

# Obtener la lista de archivos .txt en la ruta especificada
file_list = glob.glob(R_Pila_I_SIE + "/*.TXT")
file_list_IP = glob.glob(R_Pila_IP_SIE + "/*.TXT")

In [ ]:
df_pila_i_sie = pd.concat((pd.read_csv(file, sep='|', encoding='ANSI') for file in file_list), ignore_index=True)
df_pila_iP_sie = pd.concat((pd.read_csv(file, sep='|', encoding='ANSI', header=None) for file in file_list_IP), ignore_index=True)

df_Pila_3047 = pd.read_csv(R_Pila3047, sep=',', encoding='UTF-16')


In [ ]:
import pandas as pd
import gc
from typing import List

# 1. Validación de pre-condiciones y preparación de llaves
# Definición de nombres de columnas para evitar errores de typos
ID_3047: str = 'Num_rad_planilla'
ID_SIE: str = 'N° Radicación o PILA'
COLS_TO_FETCH: dict = {
    'NIT Administradora': 'Nit',
    'Razón Social Aportante': 'Razon_Soacial'
}

# 2. Deduplicación del catálogo (Source of Truth)
# Al ser una relación many-to-many, debemos reducir df_pila_i_sie a un mapeo único
# para evitar una explosión combinatoria (producto cartesiano) en el merge.
df_lookup_tmp: pd.DataFrame = df_pila_i_sie[[ID_SIE] + list(COLS_TO_FETCH.keys())].drop_duplicates(subset=[ID_SIE])

# 3. Validación de integridad: ¿Existen duplicados remanentes en la llave de cruce?
if df_lookup_tmp[ID_SIE].duplicated().any():
    # Si después de drop_duplicates persiste el duplicado, significa que un mismo ID 
    # tiene diferentes NITs o Razones Sociales. Detenemos para evitar inconsistencia.
    raise ValueError(f"Inconsistencia de datos: El ID {ID_SIE} tiene valores contradictorios en el origen.")

# 4. Operación de cruce (Left Join)
# Guardamos la cuenta original de filas para validar integridad post-merge
rows_before: int = len(df_Pila_3047)

df_Pila_3047 = df_Pila_3047.merge(
    df_lookup_tmp,
    left_on=ID_3047,
    right_on=ID_SIE,
    how='left'
)

# 5. Consolidación de datos
# Transferimos los valores recuperados a las columnas originales (asumiendo que estaban vacías/NaN)
for col_sie, col_3047 in COLS_TO_FETCH.items():
    df_Pila_3047[col_3047] = df_Pila_3047[col_sie]

# 6. Limpieza de columnas auxiliares del merge
df_Pila_3047.drop(columns=[ID_SIE] + list(COLS_TO_FETCH.keys()), inplace=True)

# 7. Validación de integridad post-merge
rows_after: int = len(df_Pila_3047)
if rows_before != rows_after:
    raise AssertionError(f"Pérdida o duplicación de datos detectada: {rows_before} != {rows_after}")

# 8. Liberación de memoria
del df_lookup_tmp
del rows_before
del rows_after
gc.collect()

In [ ]:
import pandas as pd
import gc
from typing import List

def process_pila_update(df_target: pd.DataFrame, df_source: pd.DataFrame) -> pd.DataFrame:
    """
    Actualiza Nit y Razon_Soacial en df_target usando df_source mediante Num_rad_planilla.
    Maneja la relación many-to-many reduciendo el origen a valores únicos por ID.
    """
    
    # 1. Definición de constantes de mapeo (Basado en posiciones para el origen)
    # Según instrucción: Columna 87 (Key), 4 (Nit), 2 (Razon_Social)
    KEY_TARGET: str = 'Num_rad_planilla'
    KEY_SOURCE_IDX: int = 87
    COL_NIT_IDX: int = 4
    COL_RS_IDX: int = 2
    
    # 2. Validación de pre-condiciones y dimensionalidad inicial
    initial_rows: int = df_target.shape[0]
    
    # Asegurar que las llaves sean del mismo tipo para evitar fallos en el merge
    # Se opta por string para manejar posibles ceros a la izquierda en NIT o Planilla
    df_target[KEY_TARGET] = df_target[KEY_TARGET].astype(str).str.strip()
    
    # 3. Creación de estructura de mapeo (Mapping)
    # Extraemos solo las columnas necesarias por índice para optimizar RAM
    # Eliminamos duplicados en el origen para mitigar el riesgo de explosión por M:M
    _tmp_mapping: pd.DataFrame = df_source.iloc[:, [KEY_SOURCE_IDX, COL_NIT_IDX, COL_RS_IDX]].copy()
    _tmp_mapping.columns = [KEY_TARGET, 'new_Nit', 'new_Razon_Soacial']
    _tmp_mapping[KEY_TARGET] = _tmp_mapping[KEY_TARGET].astype(str).str.strip()
    
    # Prioridad: Mantener el primer registro encontrado para cada llave única
    # Esto garantiza que el merge sea 1:1 o M:1, nunca M:M hacia el target
    _tmp_mapping = _tmp_mapping.drop_duplicates(subset=[KEY_TARGET], keep='first')
    
    # 4. Operación de Enriquecimiento (Left Join)
    # Usamos validate='many_to_one' para asegurar que el mapping no tenga duplicados
    df_target = df_target.merge(
        _tmp_mapping,
        on=KEY_TARGET,
        how='left',
        validate='m:1' 
    )
    
    # 5. Validación de integridad post-merge
    if df_target.shape[0] != initial_rows:
        # Si esto falla, la lógica de drop_duplicates en el mapping no fue suficiente o el merge falló
        raise ValueError(f"CRITICAL: Se detectó duplicidad de filas. Original: {initial_rows}, Actual: {df_target.shape[0]}")

    # 6. Actualización Idempotente
    # Llenamos vacíos solo donde las columnas originales no tienen información
    # Esto evita sobreescribir si la celda se ejecuta dos veces con datos ya cargados
    df_target['Nit'] = df_target['Nit'].fillna(df_target['new_Nit'])
    df_target['Razon_Soacial'] = df_target['Razon_Soacial'].fillna(df_target['new_Razon_Soacial'])
    
    # 7. Limpieza de columnas auxiliares
    df_target.drop(columns=['new_Nit', 'new_Razon_Soacial'], inplace=True)
    
    # 8. Liberación de memoria
    del _tmp_mapping
    gc.collect()
    
    return df_target

# Ejecución del bloque
df_Pila_3047 = process_pila_update(df_Pila_3047, df_pila_iP_sie)

In [ ]:
import pandas as pd
import gc
from typing import Tuple

def clean_and_audit_missing_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Elimina registros con valores nulos o vacíos en Nit y Razon_Soacial.
    Realiza una auditoría estricta del impacto en la dimensionalidad.
    """
    # 1. Configuración de columnas críticas
    TARGET_COLS: list[str] = ['Nit', 'Razon_Soacial']
    
    # 2. Captura de estado inicial para auditoría
    rows_before: int = len(df)
    
    # Pre-procesamiento: Convertir strings vacíos o con espacios en NaN 
    # para asegurar que dropna() los detecte correctamente.
    for col in TARGET_COLS:
        if col in df.columns:
            df[col] = df[col].replace(r'^\s*$', pd.NA, regex=True)

    # 3. Identificación de registros a eliminar (para el reporte de auditoría)
    # Se consideran registros inválidos aquellos que tengan nulo en CUALQUIERA de las dos columnas
    _mask_invalid: pd.Series = df[TARGET_COLS].isna().any(axis=1)
    rows_to_drop: int = _mask_invalid.sum()
    
    # 4. Ejecución de la limpieza
    # Creamos una copia para no afectar el objeto original de forma impredecible (Idempotencia)
    df_cleaned: pd.DataFrame = df.loc[~_mask_invalid].copy()
    
    # 5. Validación de integridad post-proceso
    rows_after: int = len(df_cleaned)
    loss_percentage: float = (rows_to_drop / rows_before) * 100 if rows_before > 0 else 0
    
    # 6. Reporte de Auditoría (Output técnico)
    print("-" * 30)
    print("REPORTE DE AUDITORÍA DE CALIDAD")
    print("-" * 30)
    print(f"Registros iniciales:      {rows_before}")
    print(f"Registros eliminados:     {rows_to_drop}")
    print(f"Registros remanentes:     {rows_after}")
    print(f"Impacto en la data:       {loss_percentage:.2f}%")
    print("-" * 30)
    
    # 7. Gestión de memoria
    del _mask_invalid
    gc.collect()
    
    return df_cleaned

# Ejecución del bloque y reasignación controlada
df_Pila_3047 = clean_and_audit_missing_data(df_Pila_3047)